In [ ]:
pip install pandas Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.1 MB/s eta 0:00:00


In [ ]:
#
# Nama File: preprocess_reviews_filled.py
# Deskripsi: Script untuk membersihkan dan memproses data ulasan wisata Sumedang,
#            termasuk mengisi ulasan kosong berdasarkan rating bintang.
#
# Instruksi untuk menjalankan:
# 1. Pastikan Anda memiliki file 'sumedang_reviews.csv' di direktori yang sama.
# 2. Install library yang dibutuhkan:
#    pip install pandas nltk Sastrawi
# 3. Jalankan script ini dari terminal:
#    python preprocess_reviews_filled.py
#

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import time

# --- Konfigurasi Awal ---
# Download daftar stopwords Bahasa Indonesia dari NLTK (hanya perlu sekali)
try:
    stopwords.words('indonesian')
except LookupError:
    print("Downloading Indonesian stopwords...")
    nltk.download('stopwords')
    print("Download complete.")

# Inisialisasi stemmer dari Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Daftar stopwords Bahasa Indonesia
indonesian_stopwords = set(stopwords.words('indonesian'))

# --- Fungsi-fungsi Preprocessing ---

def load_data(filepath):
    """
    Memuat data dari file CSV.
    """
    print(f"Memuat data dari {filepath}...")
    try:
        df = pd.read_csv(filepath)
        print("Data berhasil dimuat.")
        return df
    except FileNotFoundError:
        print(f"Error: File tidak ditemukan di '{filepath}'. Pastikan file ada di direktori yang benar.")
        return None

def fill_empty_reviews(row, mapping):
    """
    Mengisi ulasan yang kosong berdasarkan nilai rating.
    """
    review_text = row['review']
    rating = row['rating']
    # Periksa apakah review adalah string dan isinya '(No Review)' atau kosong
    if isinstance(review_text, str) and (review_text.strip().lower() == '(no review)' or review_text.strip() == ''):
        return mapping.get(rating, "") # Mengembalikan teks dari map, atau string kosong jika rating tidak ada di map
    return review_text

def clean_text(text):
    """
    Membersihkan teks dari karakter yang tidak diinginkan, mengubah ke huruf kecil.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()  # Ubah ke huruf kecil
    text = re.sub(r'http\S+', '', text)  # Hapus URL
    text = re.sub(r'[^a-z\s]', '', text)  # Hapus karakter selain huruf dan spasi
    text = text.strip()  # Hapus spasi di awal dan akhir
    return text

def remove_stopwords(text, stopwords_set):
    """
    Menghapus stopwords dari teks.
    """
    words = text.split()
    filtered_words = [word for word in words if word not in stopwords_set]
    return " ".join(filtered_words)

def stem_text(text, stemmer_instance):
    """
    Melakukan stemming pada teks.
    """
    return stemmer_instance.stem(text)

def convert_time_to_days(time_str):
    """
    Mengonversi string waktu (misal: '3 years ago') menjadi jumlah hari.
    """
    if not isinstance(time_str, str):
        return None

    time_str = time_str.lower().replace('edited ', '') # Hapus 'Edited'
    num = 1 # Default untuk 'a week', 'a month', etc.

    search_result = re.search(r'\d+', time_str)
    if search_result:
        num = int(search_result.group())

    if 'year' in time_str:
        return num * 365
    elif 'month' in time_str:
        return num * 30
    elif 'week' in time_str:
        return num * 7
    elif 'day' in time_str:
        return num
    elif 'hour' in time_str or 'minute' in time_str or 'moment' in time_str:
        return 1
    else:
        return None

def preprocess_dataframe(df):
    """
    Fungsi utama untuk menjalankan semua langkah preprocessing pada DataFrame.
    """
    print("\nMemulai proses preprocessing data...")
    start_time = time.time()

    # 1. Menangani nilai yang hilang dan duplikat
    print(f"Ukuran data awal: {df.shape[0]} baris.")
    df.dropna(subset=['review', 'rating'], inplace=True)
    df.drop_duplicates(inplace=True)

    # Pastikan 'rating' adalah tipe numerik sebelum digunakan untuk mapping
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    df.dropna(subset=['rating'], inplace=True)
    df['rating'] = df['rating'].astype(int)

    # 2. **BARU**: Mengisi ulasan '(No Review)' berdasarkan rating
    print("Mengisi ulasan '(No Review)' berdasarkan rating...")
    rating_map = {
        1: "sangat buruk",
        2: "buruk",
        3: "cukup",
        4: "baik",
        5: "sangat baik"
    }
    df['review'] = df.apply(lambda row: fill_empty_reviews(row, rating_map), axis=1)

    # Hapus baris yang review-nya mungkin masih kosong (jika ada rating di luar 1-5 atau error)
    df = df[df['review'].str.strip() != '']
    print(f"Ukuran data setelah menghapus duplikat & mengisi ulasan: {df.shape[0]} baris.")

    # 3. Memproses kolom 'review' (sekarang sudah terisi semua)
    print("Membersihkan kolom 'review'...")
    df['review_cleaned'] = df['review'].apply(clean_text)
    print("Menghapus stopwords...")
    df['review_cleaned'] = df['review_cleaned'].apply(lambda x: remove_stopwords(x, indonesian_stopwords))
    print("Melakukan stemming...")
    df['review_cleaned'] = df['review_cleaned'].apply(lambda x: stem_text(x, stemmer))

    # 4. Memproses kolom 'time'
    print("Mengonversi kolom 'time' ke jumlah hari...")
    df['days_ago'] = df['time'].apply(convert_time_to_days)

    # 5. Membuat fitur tambahan (opsional)
    df['review_length'] = df['review_cleaned'].str.len()

    # 6. Memilih dan menata ulang kolom
    final_df = df[['place', 'user', 'review', 'review_cleaned', 'rating', 'days_ago', 'review_length']]

    end_time = time.time()
    print(f"\nPreprocessing selesai dalam {end_time - start_time:.2f} detik.")

    return final_df

def save_data(df, filepath):
    """
    Menyimpan DataFrame ke file CSV.
    """
    print(f"Menyimpan data yang sudah diproses ke {filepath}...")
    df.to_csv(filepath, index=False)
    print("Data berhasil disimpan.")


# --- Eksekusi Script ---
if __name__ == "__main__":
    input_filepath = 'sumedang_reviews.csv'
    output_filepath = 'preprocessed_sumedang_reviews_filled.csv'

    reviews_df = load_data(input_filepath)

    if reviews_df is not None:
        preprocessed_df = preprocess_dataframe(reviews_df)

        print("\nContoh hasil data yang sudah diproses (termasuk yang diisi otomatis):")
        # Menampilkan contoh ulasan yang asli dan yang diisi
        print("--- Contoh Ulasan Asli ---")
        print(preprocessed_df[preprocessed_df['review'].str.lower() != '(no review)'].head(3))
        print("\n--- Contoh Ulasan yang Diisi Otomatis ---")
        print(preprocessed_df[preprocessed_df['review'].isin(["sangat buruk", "buruk", "cukup", "baik", "sangat baik"])].head(3))

        print("\nInfo data setelah diproses:")
        preprocessed_df.info()

        save_data(preprocessed_df, output_filepath)

Memuat data dari sumedang_reviews.csv...
Data berhasil dimuat.

Memulai proses preprocessing data...
Ukuran data awal: 6419 baris.
Mengisi ulasan '(No Review)' berdasarkan rating...
Ukuran data setelah menghapus duplikat & mengisi ulasan: 6382 baris.
Membersihkan kolom 'review'...
Menghapus stopwords...
Melakukan stemming...
Mengonversi kolom 'time' ke jumlah hari...

Preprocessing selesai dalam 495.79 detik.

Contoh hasil data yang sudah diproses (termasuk yang diisi otomatis):
--- Contoh Ulasan Asli ---
         place            user  \
0  Curug Sabuk        アルフィーハムザ   
1  Curug Sabuk    Tukang Ameng   
2  Curug Sabuk  Nisa Nurfariza   

                                              review  \
0                                           Matep...   
1  Start from the MTK solar cage. Accompany the g...   
2  track 3 ± hours of leisurely walking, you have...   

                                      review_cleaned  rating  days_ago  \
0                                              matep 

In [ ]:
#
# Nama File: preprocess_full_dataset.py
# Deskripsi: Script lengkap untuk memproses data ulasan individual,
#            menggabungkannya dengan data master destinasi, dan menghasilkan
#            satu dataset terpadu yang siap untuk model rekomendasi.
#
# Instruksi untuk menjalankan:
# 1. Pastikan Anda memiliki file-file berikut di direktori yang sama:
#    - 'sumedang_reviews_Alama_Budaya_Sejarah_buatan_religi_kuliner_petualangan_keluarga_kesehatan.csv' (Data ulasan utama)
#    - 'data_objek_wisata_dataset.csv' (Data master metadata destinasi)
# 2. Install library yang dibutuhkan:
#    pip install pandas nltk Sastrawi
# 3. Jalankan script ini dari terminal:
#    python preprocess_full_dataset.py
#

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import time

# --- Konfigurasi Awal ---
print("Memulai script pra-pemrosesan data...")
try:
    stopwords.words('indonesian')
except LookupError:
    print("Mengunduh stopwords Bahasa Indonesia...")
    nltk.download('stopwords')
    print("Unduhan selesai.")

print("Inisialisasi stemmer Sastrawi (mungkin butuh beberapa saat)...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()
indonesian_stopwords = set(stopwords.words('indonesian'))
print("Inisialisasi selesai.")

# --- Fungsi-fungsi Pra-pemrosesan Teks ---

def fill_empty_reviews(row, mapping):
    review_text = row['review']
    rating = row['rating']
    if pd.isna(review_text) or (isinstance(review_text, str) and review_text.strip() == ''):
        return mapping.get(rating, "")
    return review_text

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = text.strip()
    return " ".join(text.split()) # Menghapus spasi ganda

def remove_stopwords(text, stopwords_set):
    words = text.split()
    return " ".join([word for word in words if word not in stopwords_set])

def stem_text(text, stemmer_instance):
    return stemmer_instance.stem(text)

def convert_time_to_days(time_str):
    if not isinstance(time_str, str):
        return None
    time_str = time_str.lower().replace('edited', '').strip()
    num = 1
    if re.search(r'\d+', time_str):
        num = int(re.search(r'\d+', time_str).group())
    if 'tahun' in time_str:
        return num * 365
    elif 'bulan' in time_str:
        return num * 30
    elif 'minggu' in time_str:
        return num * 7
    elif 'hari' in time_str:
        return num
    elif 'jam' in time_str or 'menit' in time_str or 'detik' in time_str:
        return 1
    return None

def create_master_dataset(reviews_path, destinations_path, output_path):
    """
    Pipeline lengkap untuk memuat, memproses, dan menggabungkan data.
    """
    # 1. PROSES DATA ULASAN (DATA UTAMA)
    print("\n--- Tahap 1: Memproses Data Ulasan Individual ---")
    df_reviews = pd.read_csv(reviews_path)
    print(f"Jumlah baris ulasan awal: {len(df_reviews)}")
    df_reviews.dropna(subset=['place', 'rating'], inplace=True)
    df_reviews['rating'] = pd.to_numeric(df_reviews['rating'], errors='coerce').astype('Int64')

    rating_map = {1: "sangat buruk", 2: "buruk", 3: "cukup", 4: "baik", 5: "sangat baik"}
    df_reviews['review'] = df_reviews.apply(lambda row: fill_empty_reviews(row, rating_map), axis=1)
    df_reviews.dropna(subset=['review', 'rating'], inplace=True)

    print("Membersihkan teks ulasan (cleaning, stopwords, stemming)...")
    df_reviews['review_cleaned'] = df_reviews['review'].apply(clean_text)
    df_reviews['review_cleaned'] = df_reviews['review_cleaned'].apply(lambda x: remove_stopwords(x, indonesian_stopwords))
    df_reviews['review_cleaned'] = df_reviews['review_cleaned'].apply(lambda x: stem_text(x, stemmer))

    print("Mengonversi kolom waktu ke jumlah hari...")
    df_reviews['days_ago'] = df_reviews['time'].apply(convert_time_to_days)
    df_reviews['place_normalized'] = df_reviews['place'].str.lower().str.strip()
    print(f"Jumlah baris ulasan setelah diproses: {len(df_reviews)}")

    # 2. PROSES DATA DESTINASI (METADATA)
    print("\n--- Tahap 2: Memproses Data Master Destinasi ---")
    df_destinations = pd.read_csv(destinations_path)
    print(f"Jumlah baris destinasi awal: {len(df_destinations)}")
    df_destinations.rename(columns={'NAMA OBJEK WISATA': 'nama_wisata', 'JENIS': 'kategori'}, inplace=True)
    df_destinations['place_normalized'] = df_destinations['nama_wisata'].str.lower().str.strip()
    print("Data master destinasi siap digabungkan.")

    # 3. GABUNGKAN DATA
    print("\n--- Tahap 3: Menggabungkan Ulasan dan Destinasi ---")
    df_master = pd.merge(
        df_reviews,
        df_destinations[['place_normalized', 'kategori']], # Ambil kolom yg dibutuhkan
        on='place_normalized',
        how='left' # Jaga semua ulasan, bahkan jika destinasi tdk ada di master
    )
    unmatched_reviews = df_master['kategori'].isnull().sum()
    print(f"Penggabungan selesai. Jumlah ulasan: {len(df_master)}")
    if unmatched_reviews > 0:
        print(f"PERINGATAN: Ditemukan {unmatched_reviews} ulasan yang nama tempatnya tidak cocok dengan data master destinasi.")

    # 4. FINALISASI DATASET
    print("\n--- Tahap 4: Finalisasi Dataset Master ---")
    df_master.dropna(subset=['kategori'], inplace=True) # Hapus ulasan yg tdk cocok
    print(f"Jumlah baris setelah menghapus ulasan yang tidak cocok: {len(df_master)}")

    df_master['user_id'] = pd.factorize(df_master['user'])[0]
    df_master['destination_id'] = pd.factorize(df_master['place_normalized'])[0]

    final_columns = [
        'user_id', 'destination_id', 'user', 'place', 'kategori', 'rating',
        'review', 'review_cleaned', 'days_ago', 'time'
    ]
    df_master = df_master[final_columns]

    print("Dataset master berhasil dibuat dan difinalisasi.")
    df_master.to_csv(output_path, index=False)
    print(f"Dataset master telah disimpan di '{output_path}'")
    return df_master

if __name__ == "__main__":
    start_time = time.time()

    # Definisikan path file
    reviews_file = 'sumedang_reviews_Alama_Budaya_Sejarah_buatan_religi_kuliner_petualangan_keluarga_kesehatan.csv'
    destinations_file = 'data_objek_wisata_dataset.csv'
    output_file = 'master_dataset.csv'

    # Jalankan pipeline
    master_df = create_master_dataset(reviews_file, destinations_file, output_file)

    # Tampilkan info dan contoh hasil
    if master_df is not None:
        print("\n--- INFO DATASET MASTER FINAL ---")
        master_df.info()
        print("\n--- CONTOH 5 BARIS PERTAMA DATASET MASTER ---")
        print(master_df.head())

    end_time = time.time()
    print(f"\nTotal waktu eksekusi: {end_time - start_time:.2f} detik.")



Memulai script pra-pemrosesan data...
Inisialisasi stemmer Sastrawi (mungkin butuh beberapa saat)...
Inisialisasi selesai.

--- Tahap 1: Memproses Data Ulasan Individual ---
Jumlah baris ulasan awal: 45775
Membersihkan teks ulasan (cleaning, stopwords, stemming)...
Mengonversi kolom waktu ke jumlah hari...
Jumlah baris ulasan setelah diproses: 45775

--- Tahap 2: Memproses Data Master Destinasi ---
Jumlah baris destinasi awal: 75
Data master destinasi siap digabungkan.

--- Tahap 3: Menggabungkan Ulasan dan Destinasi ---
Penggabungan selesai. Jumlah ulasan: 45775
PERINGATAN: Ditemukan 43024 ulasan yang nama tempatnya tidak cocok dengan data master destinasi.

--- Tahap 4: Finalisasi Dataset Master ---
Jumlah baris setelah menghapus ulasan yang tidak cocok: 2751
Dataset master berhasil dibuat dan difinalisasi.
Dataset master telah disimpan di 'master_dataset.csv'

--- INFO DATASET MASTER FINAL ---
<class 'pandas.core.frame.DataFrame'>
Index: 2751 entries, 0 to 44855
Data columns (total 

In [ ]:
#
# Nama File: explore_dataset.py
# Deskripsi: Script untuk melakukan analisis data eksploratif (EDA) pada
#            master_dataset.csv guna memahami karakteristik dan distribusi data.
#
# Instruksi untuk menjalankan:
# 1. Pastikan Anda memiliki file 'master_dataset.csv' di direktori yang sama.
# 2. Install library yang dibutuhkan:
#    pip install pandas
# 3. Jalankan script ini dari terminal:
#    python explore_dataset.py
#

import pandas as pd

def analyze_dataset(filepath):
    """
    Fungsi utama untuk memuat dan menganalisis dataset master.
    """
    print(f"Membaca dataset dari '{filepath}'...")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"ERROR: File tidak ditemukan di '{filepath}'. Mohon periksa kembali nama dan lokasi file.")
        return

    print("\n" + "="*50)
    print("ANALISIS DATA EKSPLORATIF - DATASET WISATA SUMEDANG")
    print("="*50)

    # 1. Informasi Dasar Dataset
    print("\n[1] INFORMASI DASAR & STRUKTUR DATA")
    print("-" * 35)
    df.info()

    # 2. Contoh 5 Baris Pertama Data
    print("\n[2] CONTOH 5 BARIS PERTAMA DATA")
    print("-" * 35)
    print(df.head())

    # 3. Statistik Deskriptif untuk Kolom Numerik
    print("\n[3] STATISTIK DESKRIPTIF (Rating & Days Ago)")
    print("-" * 50)
    # Memilih kolom numerik yang relevan untuk dianalisis
    print(df[['rating', 'days_ago']].describe())

    # 4. Distribusi Kategori Wisata
    print("\n[4] DISTRIBUSI KATEGORI WISATA (PENTING UNTUK METRIK DIVERSITY)")
    print("-" * 65)
    category_counts = df['kategori'].value_counts()
    print(category_counts)
    print(f"\n-> Terdapat {len(category_counts)} kategori unik.")
    print(f"-> Kategori paling umum: '{category_counts.index[0]}'")

    # 5. Distribusi Rating
    print("\n[5] DISTRIBUSI RATING")
    print("-" * 25)
    rating_counts = df['rating'].value_counts().sort_index(ascending=False)
    print(rating_counts)
    print("\n-> Mayoritas ulasan memberikan rating bintang 5.")

    # 6. Destinasi Paling Populer (Berdasarkan Jumlah Ulasan)
    print("\n[6] 10 DESTINASI PALING POPULER (Berdasarkan Jumlah Ulasan)")
    print("-" * 65)
    popular_places = df['place'].value_counts().head(10)
    print(popular_places)
    print("\n-> Analisis ini penting untuk mengidentifikasi 'bias popularitas' yang ingin diatasi dalam tesis Anda.")

    # 7. 10 Pengguna Paling Aktif (Berdasarkan Jumlah Ulasan)
    print("\n[7] 10 PENGGUNA PALING AKTIF")
    print("-" * 30)
    active_users = df['user'].value_counts().head(10)
    print(active_users)

if __name__ == "__main__":
    # Ganti nama file ini jika nama file Anda berbeda
    master_dataset_path = 'master_dataset.csv'
    analyze_dataset(master_dataset_path)


Membaca dataset dari 'master_dataset.csv'...

ANALISIS DATA EKSPLORATIF - DATASET WISATA SUMEDANG

[1] INFORMASI DASAR & STRUKTUR DATA
-----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2751 entries, 0 to 2750
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   user_id         2751 non-null   int64 
 1   destination_id  2751 non-null   int64 
 2   user            2751 non-null   object
 3   place           2751 non-null   object
 4   kategori        2751 non-null   object
 5   rating          2751 non-null   int64 
 6   review          2751 non-null   object
 7   review_cleaned  2726 non-null   object
 8   days_ago        2751 non-null   int64 
 9   time            2751 non-null   object
dtypes: int64(4), object(6)
memory usage: 215.1+ KB

[2] CONTOH 5 BARIS PERTAMA DATA
-----------------------------------
   user_id  destination_id                     user          place  \
0  

In [ ]:
#
# Nama File: normalize_dataset.py
# Deskripsi: Script untuk memecah 'master_dataset.csv' menjadi beberapa file CSV
#            terstruktur (Destinations, Users, Ratings, Reviews, Categories)
#            yang menyerupai skema database relasional.
#
# Instruksi untuk menjalankan:
# 1. Pastikan Anda memiliki file-file berikut di direktori yang sama:
#    - 'master_dataset.csv'
#    - 'data_objek_wisata_dataset.csv' (sebagai sumber metadata detail destinasi)
# 2. Install library yang dibutuhkan:
#    pip install pandas
# 3. Jalankan script ini dari terminal:
#    python normalize_dataset.py
#

import pandas as pd
import os

def normalize_master_dataset(master_path, destinations_metadata_path):
    """
    Memuat dataset master dan memecahnya menjadi beberapa file CSV yang ternormalisasi.
    """
    print(f"Membaca dataset utama dari '{master_path}'...")
    try:
        df_master = pd.read_csv(master_path)
        df_dest_meta = pd.read_csv(destinations_metadata_path)
    except FileNotFoundError as e:
        print(f"ERROR: File tidak ditemukan. Pastikan file berikut ada:\n- {master_path}\n- {destinations_metadata_path}")
        print(f"Detail error: {e}")
        return

    print("Dataset berhasil dimuat. Memulai proses normalisasi...")
    output_dir = "normalized_data"
    os.makedirs(output_dir, exist_ok=True)
    print(f"File output akan disimpan di folder '{output_dir}/'")

    # --- 1. Membuat Tabel Categories ---
    print("[1] Membuat Categories.csv...")
    unique_categories = df_master['kategori'].unique()
    df_categories = pd.DataFrame({
        'category_id': range(len(unique_categories)),
        'name': unique_categories
    })
    df_categories.to_csv(os.path.join(output_dir, 'Categories.csv'), index=False)
    print("-> Selesai.")

    # Buat mapping dari nama kategori ke ID untuk digunakan nanti
    category_to_id = df_categories.set_index('name')['category_id'].to_dict()

    # --- 2. Membuat Tabel Destinations ---
    print("[2] Membuat Destinations.csv...")
    # Ambil destinasi unik dari master dataset
    df_destinations = df_master[['destination_id', 'place', 'kategori']].drop_duplicates().reset_index(drop=True)
    df_destinations['category_id'] = df_destinations['kategori'].map(category_to_id)

    # Normalisasi nama di metadata untuk penggabungan
    df_dest_meta['place_normalized'] = df_dest_meta['NAMA OBJEK WISATA'].str.lower().str.strip()
    df_destinations['place_normalized'] = df_destinations['place'].str.lower().str.strip()

    # Gabungkan untuk mendapatkan deskripsi dan alamat
    df_destinations = pd.merge(
        df_destinations,
        df_dest_meta[['place_normalized', 'DESKRIPSI OBJEK WISATA', 'ALAMAT/LOKASI']],
        on='place_normalized',
        how='left'
    )

    # Tambahkan kolom placeholder yang tidak ada
    df_destinations['latitude'] = None
    df_destinations['longitude'] = None

    # Pilih, ganti nama, dan urutkan kolom sesuai permintaan
    df_destinations = df_destinations.rename(columns={
        'place': 'name',
        'DESKRIPSI OBJEK WISATA': 'description',
        'ALAMAT/LOKASI': 'address'
    })
    final_destinations = df_destinations[['destination_id', 'name', 'description', 'latitude', 'longitude', 'address', 'category_id']]
    final_destinations.to_csv(os.path.join(output_dir, 'Destinations.csv'), index=False)
    print("-> Selesai.")

    # --- 3. Membuat Tabel Users ---
    print("[3] Membuat Users.csv...")
    df_users = df_master[['user_id', 'user']].drop_duplicates().reset_index(drop=True)
    df_users['email'] = None
    df_users['preferences'] = None

    df_users = df_users.rename(columns={'user': 'name'})
    final_users = df_users[['user_id', 'name', 'email', 'preferences']]
    final_users.to_csv(os.path.join(output_dir, 'Users.csv'), index=False)
    print("-> Selesai.")

    # --- 4. Membuat Tabel Ratings ---
    print("[4] Membuat Ratings.csv...")
    df_ratings = df_master[['user_id', 'destination_id', 'rating', 'time']].copy()
    df_ratings['rating_id'] = df_ratings.index

    df_ratings = df_ratings.rename(columns={'time': 'timestamp'})
    final_ratings = df_ratings[['rating_id', 'user_id', 'destination_id', 'rating', 'timestamp']]
    final_ratings.to_csv(os.path.join(output_dir, 'Ratings.csv'), index=False)
    print("-> Selesai.")

    # --- 5. Membuat Tabel Reviews ---
    print("[5] Membuat Reviews.csv...")
    df_reviews = df_master[['user_id', 'destination_id', 'review', 'time']].copy()
    df_reviews['review_id'] = df_reviews.index
    df_reviews['title'] = None

    df_reviews = df_reviews.rename(columns={'review': 'content', 'time': 'timestamp'})
    final_reviews = df_reviews[['review_id', 'user_id', 'destination_id', 'title', 'content', 'timestamp']]
    final_reviews.to_csv(os.path.join(output_dir, 'Reviews.csv'), index=False)
    print("-> Selesai.")

    print("\nProses normalisasi selesai. Semua file telah disimpan di folder 'normalized_data'.")


if __name__ == "__main__":
    # Path ke file input Anda
    master_file_path = 'master_dataset.csv'
    destinations_metadata_file_path = 'data_objek_wisata_dataset.csv'

    normalize_master_dataset(master_file_path, destinations_metadata_file_path)



Membaca dataset utama dari 'master_dataset.csv'...
Dataset berhasil dimuat. Memulai proses normalisasi...
File output akan disimpan di folder 'normalized_data/'
[1] Membuat Categories.csv...
-> Selesai.
[2] Membuat Destinations.csv...
-> Selesai.
[3] Membuat Users.csv...
-> Selesai.
[4] Membuat Ratings.csv...
-> Selesai.
[5] Membuat Reviews.csv...
-> Selesai.

Proses normalisasi selesai. Semua file telah disimpan di folder 'normalized_data'.
